# DeepTruth — Retrain with More Real Phone Photos
Phase 1 only (stable 98%). Adds CelebA + LFW real photos to fix false positives on phone/WhatsApp images.

In [ ]:
# ── 1. Mount Drive & setup ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
DRIVE_DIR = '/content/drive/MyDrive/DeepTruth'
os.makedirs(DRIVE_DIR, exist_ok=True)

# Copy model_arch.py from Drive
shutil.copy(f'{DRIVE_DIR}/model_arch.py', '/content/model_arch.py')
print('model_arch.py copied')

In [ ]:
# ── 2. Install packages ────────────────────────────────────────────────
!pip install -q timm transformers kaggle

In [ ]:
# ── 3. Kaggle credentials ──────────────────────────────────────────────
from google.colab import files
print('Upload your kaggle.json')
files.upload()
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
print('Kaggle ready')

In [ ]:
# ── 4. Download real phone photo datasets ─────────────────────────────
import os
os.makedirs('/content/real_extra', exist_ok=True)

# CelebA — 200k+ diverse celebrity/phone photos (~1.4 GB)
print('Downloading CelebA...')
!kaggle datasets download -d jessicali9530/celeba-dataset -p /content/celeba --unzip -q
print('CelebA done')

# LFW — 13k real face photos from news/web (~170 MB)
print('Downloading LFW...')
!kaggle datasets download -d atulanandjha/lfwpeople -p /content/lfw --unzip -q
print('LFW done')

In [ ]:
# ── 5. Collect all real images ─────────────────────────────────────────
import glob, random
from pathlib import Path

EXTS = {'.jpg', '.jpeg', '.png'}

def collect_images(folder, max_count=None):
    imgs = [p for p in Path(folder).rglob('*') if p.suffix.lower() in EXTS]
    random.shuffle(imgs)
    return [str(p) for p in imgs[:max_count]] if max_count else [str(p) for p in imgs]

# CelebA images
celeba_imgs = collect_images('/content/celeba', max_count=30000)
print(f'CelebA: {len(celeba_imgs)} images')

# LFW images
lfw_imgs = collect_images('/content/lfw', max_count=10000)
print(f'LFW: {len(lfw_imgs)} images')

# Existing real images from Drive (from previous training)
existing_real = []
for folder in ['real', 'Real', 'real_images', 'extra_real']:
    p = f'{DRIVE_DIR}/data/{folder}'
    if os.path.exists(p):
        existing_real += collect_images(p)
        print(f'Found existing real: {p} ({len(collect_images(p))} imgs)')

all_real = celeba_imgs + lfw_imgs + existing_real
print(f'\nTotal real images: {len(all_real)}')

In [ ]:
# ── 6. Collect all fake images from Drive ─────────────────────────────
FAKE_FOLDERS = [
    'Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures',
    'DeepFakeDetection', 'FaceShifter', 'GAN', 'Diffusion',
    'dalle', 'DALLE', 'midjourney', 'Midjourney',
    'StyleGAN', 'stylegan', 'DeepFaceLab', 'deepfacelab',
    'stable_diffusion', 'StableDiffusion', 'fake', 'fakes', 'extra_fakes'
]

all_fake = []
for folder in FAKE_FOLDERS:
    p = f'{DRIVE_DIR}/data/{folder}'
    if os.path.exists(p):
        imgs = collect_images(p)
        all_fake += imgs
        print(f'{folder}: {len(imgs)} images')

# Also scan data/ root for any missed fake subfolders
data_root = f'{DRIVE_DIR}/data'
if os.path.exists(data_root):
    for d in os.listdir(data_root):
        full = os.path.join(data_root, d)
        if os.path.isdir(full) and d.lower() not in ['real', 'real_images', 'extra_real'] and full not in [f'{DRIVE_DIR}/data/{f}' for f in FAKE_FOLDERS]:
            imgs = collect_images(full)
            if imgs:
                all_fake += imgs
                print(f'Auto-found {d}: {len(imgs)} images')

print(f'\nTotal fake images: {len(all_fake)}')

In [ ]:
# ── 7. Dataset & DataLoader ────────────────────────────────────────────
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
import cv2
import numpy as np
from PIL import Image

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.1),
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class FaceDataset(Dataset):
    def __init__(self, real_paths, fake_paths, transform=None):
        # label: 0=real, 1=fake  (probs[1] = fake at inference)
        self.items = [(p, 0) for p in real_paths] + [(p, 1) for p in fake_paths]
        random.shuffle(self.items)
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        path, label = self.items[idx]
        try:
            img = Image.open(path).convert('RGB')
            if self.transform:
                img = self.transform(img)
            return img, label
        except Exception:
            # Return a blank tensor if image is corrupt
            return torch.zeros(3, 224, 224), label

random.seed(42)

# 90/10 train/val split
def split(lst, ratio=0.9):
    n = int(len(lst) * ratio)
    return lst[:n], lst[n:]

real_train, real_val = split(all_real)
fake_train, fake_val = split(all_fake)

train_ds = FaceDataset(real_train, fake_train, transform=train_tf)
val_ds   = FaceDataset(real_val,   fake_val,   transform=val_tf)

# Weighted sampler for 50/50 real/fake balance
labels = [item[1] for item in train_ds.items]
n_real = labels.count(0)
n_fake = labels.count(1)
weights = [1.0/n_real if l == 0 else 1.0/n_fake for l in labels]
sampler = WeightedRandomSampler(weights, num_samples=min(len(labels), 60000), replacement=True)

train_loader = DataLoader(train_ds, batch_size=32, sampler=sampler,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)} ({n_real} real, {n_fake} fake)')
print(f'Val:   {len(val_ds)}')

In [ ]:
# ── 8. Load existing checkpoint ────────────────────────────────────────
import sys
sys.path.insert(0, '/content')
from model_arch import DeepTruthHybridV2, DeepTruthLoss

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

CKPT_PATH = f'{DRIVE_DIR}/deeptruth_image_model_final.pth'

ckpt = torch.load(CKPT_PATH, map_location='cpu')
num_fake_types = ckpt.get('num_fake_types', 10)

torch.manual_seed(42)
model = DeepTruthHybridV2(num_fake_types=num_fake_types, dropout=0.3)

# Remap old key names if needed
remap = {
    'stream1_clip': 'clip_stream', 'stream2_effnet': 'effnet_stream',
    'stream3_freq': 'freq_stream', 'stream4_srm': 'srm_stream',
    'stream5_gram': 'gram_stream', 'fusion': 'image_fusion',
    'head': 'image_head', 'binary_out': 'image_binary_out', 'type_out': 'image_type_out',
}
state = {}
for k, v in ckpt['model_state'].items():
    new_k = k
    for old, new in remap.items():
        if k.startswith(old + '.'):
            new_k = new + k[len(old):]
            break
    state[new_k] = v

missing, unexpected = model.load_state_dict(state, strict=False)
print(f'Loaded checkpoint: {len(state) - len(unexpected)}/{len(state)} keys')
print(f'Previous metrics: {ckpt.get("metrics", {})}')
model = model.to(DEVICE)

In [ ]:
# ── 9. Phase 1 setup — freeze backbones, train heads only ─────────────
# Freeze CLIP encoder layers 0-7
for i, layer in enumerate(model.clip_stream.clip.vision_model.encoder.layers):
    for p in layer.parameters():
        p.requires_grad = (i >= 8)
for p in model.clip_stream.clip.vision_model.embeddings.parameters():
    p.requires_grad = False

# Freeze EfficientNet blocks 0-3
for name, param in model.effnet_stream.backbone.named_parameters():
    param.requires_grad = not any(f'blocks.{i}.' in name for i in range(4))

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')

criterion = DeepTruthLoss(type_weight=0.3, gamma=2.0, label_smoothing=0.1)
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=5e-4, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5, eta_min=1e-5)

In [ ]:
# ── 10. Training loop ─────────────────────────────────────────────────
import time

EPOCHS = 5
best_avg = ckpt.get('metrics', {}).get('avg_acc', 0.0)
best_ckpt_path = f'{DRIVE_DIR}/deeptruth_image_model_final.pth'
print(f'Starting from best_avg={best_avg:.4f}\n')

def evaluate(loader):
    model.eval()
    correct_real = total_real = correct_fake = total_fake = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out = model(imgs)
            preds = out['fake_logit'].argmax(dim=1)
            real_mask = labels == 0
            fake_mask = labels == 1
            correct_real += (preds[real_mask] == 0).sum().item()
            total_real   += real_mask.sum().item()
            correct_fake += (preds[fake_mask] == 1).sum().item()
            total_fake   += fake_mask.sum().item()
    real_acc = correct_real / max(total_real, 1)
    fake_acc = correct_fake / max(total_fake, 1)
    return real_acc, fake_acc, (real_acc + fake_acc) / 2

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    t0 = time.time()

    for batch_idx, (imgs, labels) in enumerate(train_loader):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()

        out = model(imgs)
        # type_ids: real=0, fake=9 (Unknown fake) for extra real data
        type_ids = torch.where(labels == 0, torch.zeros_like(labels), torch.full_like(labels, 9))
        loss = criterion(out['binary_logit'], labels, out['type_logits'], type_ids)

        if torch.isnan(loss):
            print(f'  NaN loss at batch {batch_idx}, skipping')
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()

        if (batch_idx + 1) % 50 == 0:
            print(f'  Epoch {epoch} [{batch_idx+1}/{len(train_loader)}] loss={loss.item():.4f}')

    scheduler.step()
    real_acc, fake_acc, avg_acc = evaluate(val_loader)
    elapsed = time.time() - t0
    print(f'Epoch {epoch}/{EPOCHS} | loss={total_loss/len(train_loader):.4f} | real={real_acc*100:.1f}% fake={fake_acc*100:.1f}% avg={avg_acc*100:.1f}% | {elapsed:.0f}s')

    if avg_acc > best_avg:
        best_avg = avg_acc
        torch.save({
            'model_state': model.state_dict(),
            'num_fake_types': num_fake_types,
            'fake_type_map': ckpt.get('fake_type_map', {}),
            'epoch': epoch,
            'metrics': {'real_acc': real_acc, 'fake_acc': fake_acc, 'avg_acc': avg_acc},
        }, best_ckpt_path)
        print(f'  ✓ Saved new best checkpoint (avg={avg_acc*100:.1f}%)')

print(f'\nDone. Best avg accuracy: {best_avg*100:.1f}%')
print(f'Model saved to: {best_ckpt_path}')

In [ ]:
# ── 11. Verify saved checkpoint ────────────────────────────────────────
saved = torch.load(best_ckpt_path, map_location='cpu')
print('Saved checkpoint metrics:', saved['metrics'])
print('Epoch:', saved['epoch'])
print('Keys in checkpoint:', list(saved.keys()))
size_mb = os.path.getsize(best_ckpt_path) / 1024 / 1024
print(f'File size: {size_mb:.1f} MB')